In [1]:
%pip install chronos-forecasting
%pip install transformers accelerate

In [2]:
import pandas as pd
import numpy as np
from chronos import Chronos2Pipeline
from sklearn.metrics import r2_score,mean_squared_error

In [3]:
df = pd.read_csv("../data/training/skopje_final_data.csv",low_memory=False)
df.head()

,timestamp,sensorId,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,...,neighbor5_wind_speed,season,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2023-12-01 00:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,75.916667,NaN,NaN,981.158046,6.359756,7.391761,74.00,75.916667,...,11.246759,winter,0.000000,1.000000,-0.5,0.866025,-0.433884,-0.900969,0,1
1,2023-12-01 01:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,76.331140,NaN,NaN,981.250000,6.315041,7.238935,73.75,76.331140,...,12.313894,winter,0.258819,0.965926,-0.5,0.866025,-0.433884,-0.900969,0,1
2,2023-12-01 02:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,76.574786,NaN,NaN,981.250000,6.236111,7.570863,74.50,76.574786,...,10.972620,winter,0.500000,0.866025,-0.5,0.866025,-0.433884,-0.900969,0,1
3,2023-12-01 03:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,76.818376,NaN,NaN,981.133333,6.378968,7.031864,75.75,76.818376,...,10.233123,winter,0.707107,0.707107,-0.5,0.866025,-0.433884,-0.900969,0,1
4,2023-12-01 04:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,77.098291,NaN,NaN,981.044444,6.593254,6.751236,73.75,77.098291,...,10.739832,winter,0.866025,0.500000,-0.5,0.866025,-0.433884,-0.900969,0,1


In [4]:
df.describe()

,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor4_humidity,...,neighbor4_wind_speed,neighbor5_wind_speed,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
count,1.315275e+06,1.017354e+06,1.014088e+06,1.315275e+06,1.315275e+06,1.315800e+06,1.315275e+06,1.315275e+06,1.315275e+06,1.315275e+06,...,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06,1.315800e+06
mean,5.566136e+01,2.492260e+01,1.387223e+01,9.828889e+02,1.720202e+01,4.959989e+00,5.530974e+01,5.483988e+01,5.550372e+01,5.499049e+01,...,4.848925e+00,4.847676e+00,-1.850270e-17,-5.551014e-17,-2.785087e-03,-3.554140e-03,-2.996776e-03,-6.839945e-04,2.872777e-01,4.145007e-01
std,1.782989e+01,4.210747e+01,2.289562e+01,9.442534e+00,9.319345e+00,3.199850e+00,1.748186e+01,1.797581e+01,1.810821e+01,1.741685e+01,...,2.876215e+00,2.877265e+00,7.071070e-01,7.071070e-01,7.068597e-01,7.073399e-01,7.073425e-01,7.068648e-01,4.524924e-01,4.926358e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,9.220000e+02,-5.800000e+01,0.000000e+00,0.000000e+00,4.500000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-9.749279e-01,-9.009689e-01,0.000000e+00,0.000000e+00
25%,4.185390e+01,6.000000e+00,3.250000e+00,9.787500e+02,9.703252e+00,2.811690e+00,4.175000e+01,4.100000e+01,4.141880e+01,4.150000e+01,...,2.833955e+00,2.817445e+00,-7.071068e-01,-7.071068e-01,-5.000000e-01,-8.660254e-01,-7.818315e-01,-9.009689e-01,0.000000e+00,0.000000e+00
50%,5.575000e+01,1.200000e+01,6.750000e+00,9.830000e+02,1.641870e+01,4.510787e+00,5.536111e+01,5.450000e+01,5.537326e+01,5.525000e+01,...,4.503598e+00,4.503598e+00,6.123234e-17,-6.123234e-17,0.000000e+00,-1.836970e-16,0.000000e+00,-2.225209e-01,0.000000e+00,0.000000e+00
75%,6.950000e+01,2.766667e+01,1.533333e+01,9.880000e+02,2.432623e+01,6.432324e+00,6.900000e+01,6.850000e+01,6.964757e+01,6.867029e+01,...,6.341009e+00,6.359372e+00,7.071068e-01,7.071068e-01,8.660254e-01,5.000000e-01,7.818315e-01,6.234898e-01,1.000000e+00,1.000000e+00
max,9.900000e+01,1.873750e+03,9.980000e+02,1.195000e+03,5.950000e+01,4.566469e+01,9.900000e+01,9.900000e+01,9.900000e+01,9.900000e+01,...,2.926869e+01,2.926869e+01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,9.749279e-01,1.000000e+00,1.000000e+00,1.000000e+00


In [5]:
len(df)

1315800

In [6]:
df['sensorId'].value_counts()

sensorId
007f2b03-94e6-47b3-9e3e-44273354acd5    17544
sensor_dev_62788_603                    17544
sensor_dev_39548_976                    17544
sensor_dev_34617_228                    17544
sensor_dev_20701_157                    17544
                                        ...  
5de2a490-05f1-4f33-927e-a7e6f5664b72    17544
59c15198-7ba8-45af-aac4-5b3d1560fd81    17544
52546b6e-77bf-40f8-b0e5-6e54d7947e9f    17544
3e2465de-c2c2-4473-9c62-7113265debd9    17544
sensor_dev_84941_208                    17544
Name: count, Length: 75, dtype: int64

In [7]:
TARGET = 'pm25'
ID_COL = 'sensorId'
TIME_COL = 'timestamp'
PREDICTION_LENGTH = 512

In [8]:
df[TIME_COL] = pd.to_datetime(df[TIME_COL])

In [9]:
df

,timestamp,sensorId,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,...,neighbor5_wind_speed,season,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2023-12-01 00:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,75.916667,NaN,NaN,981.158046,6.359756,7.391761,74.00,75.916667,...,11.246759,winter,0.000000,1.000000,-0.500000,0.866025,-0.433884,-0.900969,0,1
1,2023-12-01 01:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,76.331140,NaN,NaN,981.250000,6.315041,7.238935,73.75,76.331140,...,12.313894,winter,0.258819,0.965926,-0.500000,0.866025,-0.433884,-0.900969,0,1
2,2023-12-01 02:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,76.574786,NaN,NaN,981.250000,6.236111,7.570863,74.50,76.574786,...,10.972620,winter,0.500000,0.866025,-0.500000,0.866025,-0.433884,-0.900969,0,1
3,2023-12-01 03:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,76.818376,NaN,NaN,981.133333,6.378968,7.031864,75.75,76.818376,...,10.233123,winter,0.707107,0.707107,-0.500000,0.866025,-0.433884,-0.900969,0,1
4,2023-12-01 04:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,77.098291,NaN,NaN,981.044444,6.593254,6.751236,73.75,77.098291,...,10.739832,winter,0.866025,0.500000,-0.500000,0.866025,-0.433884,-0.900969,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1315795,2025-11-30 19:00:00+00:00,sensor_dev_84941_208,83.000000,NaN,NaN,981.371212,7.000000,0.663094,80.75,74.500000,...,0.254558,autumn,-0.965926,0.258819,-0.866025,0.500000,-0.781831,0.623490,1,1
1315796,2025-11-30 20:00:00+00:00,sensor_dev_84941_208,85.500000,NaN,NaN,981.377604,6.750000,1.077217,82.25,78.500000,...,0.360000,autumn,-0.866025,0.500000,-0.866025,0.500000,-0.781831,0.623490,1,1
1315797,2025-11-30 21:00:00+00:00,sensor_dev_84941_208,88.000000,NaN,NaN,981.609375,6.000000,0.762645,83.00,81.000000,...,0.254558,autumn,-0.707107,0.707107,-0.866025,0.500000,-0.781831,0.623490,1,1
1315798,2025-11-30 22:00:00+00:00,sensor_dev_84941_208,89.500000,NaN,NaN,981.594086,6.000000,0.961617,84.25,81.000000,...,0.648999,autumn,-0.500000,0.866025,-0.866025,0.500000,-0.781831,0.623490,1,1


In [10]:
context_df = df.groupby(ID_COL).apply(lambda x: x.iloc[:-PREDICTION_LENGTH]).reset_index(drop=True)
ground_truth_df = df.groupby(ID_COL).apply(lambda x: x.iloc[-PREDICTION_LENGTH:]).reset_index(drop=True)

/tmp/ipykernel_356086/2320177275.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  context_df = df.groupby(ID_COL).apply(lambda x: x.iloc[:-PREDICTION_LENGTH]).reset_index(drop=True)
/tmp/ipykernel_356086/2320177275.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ground_truth_df = df.groupby(ID_COL).apply(lambda x: x.iloc[-PREDICTION_LENGTH:]).reset_index(drop=True)


In [11]:
ground_truth_df

,timestamp,sensorId,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,...,neighbor5_wind_speed,season,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2025-11-09 16:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,84.00,5.750000,3.50,945.000000,13.25,4.311284,84.294160,84.294160,...,3.710795,autumn,-0.866025,-5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
1,2025-11-09 17:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,84.25,4.250000,2.00,945.000000,13.00,2.845667,85.533274,85.533274,...,2.106846,autumn,-0.965926,-2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
2,2025-11-09 18:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,85.00,3.333333,2.00,945.000000,13.00,5.011445,85.398099,85.398099,...,3.784283,autumn,-1.000000,-1.836970e-16,-0.866025,0.5,-0.781831,0.62349,1,1
3,2025-11-09 19:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,86.25,3.750000,1.75,945.000000,13.00,4.390272,84.552909,84.552909,...,3.572898,autumn,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
4,2025-11-09 20:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,87.00,3.250000,2.25,945.000000,13.00,3.476920,85.398771,85.398771,...,3.244996,autumn,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38395,2025-11-30 19:00:00+00:00,sensor_dev_84941_208,83.00,NaN,NaN,981.371212,7.00,0.663094,80.750000,74.500000,...,0.254558,autumn,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
38396,2025-11-30 20:00:00+00:00,sensor_dev_84941_208,85.50,NaN,NaN,981.377604,6.75,1.077217,82.250000,78.500000,...,0.360000,autumn,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
38397,2025-11-30 21:00:00+00:00,sensor_dev_84941_208,88.00,NaN,NaN,981.609375,6.00,0.762645,83.000000,81.000000,...,0.254558,autumn,-0.707107,7.071068e-01,-0.866025,0.5,-0.781831,0.62349,1,1
38398,2025-11-30 22:00:00+00:00,sensor_dev_84941_208,89.50,NaN,NaN,981.594086,6.00,0.961617,84.250000,81.000000,...,0.648999,autumn,-0.500000,8.660254e-01,-0.866025,0.5,-0.781831,0.62349,1,1


In [12]:
forecast_df = pd.read_csv('../data/raw/skopje_forecast_weather.csv')
forecast_df

,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure
0,2025-11-09 16:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.300,89.000000,2.200000,NaN,948.50000
1,2025-11-09 17:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.000,90.000000,3.600000,NaN,948.40000
2,2025-11-09 18:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.600,92.000000,2.200000,NaN,947.90000
3,2025-11-09 19:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.400,94.000000,1.800000,NaN,948.20000
4,2025-11-09 20:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.300,95.000000,2.800000,NaN,947.80000
...,...,...,...,...,...,...,...,...,...
501079,2026-03-01 17:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,5.438,71.722170,1.548418,324.462250,995.15894
501080,2026-03-01 18:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,7.638,66.715480,1.310420,15.945477,995.11540
501081,2026-03-01 19:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.788,69.456400,2.305125,308.659820,995.31110
501082,2026-03-01 20:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.338,68.128060,1.913426,318.814150,995.45435


In [13]:
neighbourhood_matrix = pd.read_csv('../data/neighbors_data/skopje_neighbors.csv')
valid_sensors = df['sensorId'].unique()
neighbors_df_clean = neighbourhood_matrix[
    (neighbourhood_matrix['sensor_id'].isin(valid_sensors)) &
    (neighbourhood_matrix['neighbor_id'].isin(valid_sensors))
].copy()

In [14]:
def append_neighbors(df_hourly, neighbors_df, weather_cols, k_search=20, k_keep=5):
    # 1. Standardize the neighbor list
    # Ensure we only take the top K based on distance
    neighbors_topk = (
        neighbors_df.sort_values(["sensor_id", "distance_km"])
        .groupby("sensor_id")
        .head(k_search)
        .copy()
    )

    # Track original distance rank
    neighbors_topk['dist_rank'] = neighbors_topk.groupby("sensor_id").cumcount() + 1

    # 2. Merge with main data
    # We use 'neighbor_id' from the matrix to match 'sensorId' in the hourly data
    neighbor_values = neighbors_topk.merge(
        df_hourly[['sensorId', 'timestamp'] + weather_cols],
        left_on='neighbor_id',
        right_on='sensorId',
        how='inner'
    )

    # 3. Filter for availability
    # The 'sensor_id' here is the ORIGINAL sensor we are finding neighbors for
    available_topk = (
        neighbor_values.sort_values(['sensor_id', 'timestamp', 'dist_rank'])
        .groupby(['sensor_id', 'timestamp'])
        .head(k_keep)
        .copy()
    )

    # Create the 1, 2, 3 rank for the wide-format columns
    available_topk['final_rank'] = available_topk.groupby(['sensor_id', 'timestamp']).cumcount() + 1

    # 4. Pivot to wide format
    pivot_df = available_topk.pivot(
        index=['sensor_id', 'timestamp'],
        columns='final_rank',
        values=weather_cols
    )

    # Clean up column names: neighbor1_temp, neighbor2_temp, etc.
    if isinstance(pivot_df.columns, pd.MultiIndex):
        pivot_df.columns = [f"neighbor{rank}_{col}" for col, rank in pivot_df.columns]
    else:
        # Handle case with only one weather column
        pivot_df.columns = [f"neighbor{i}_{weather_cols[0]}" for i in pivot_df.columns]

    pivot_df = pivot_df.reset_index()

    # 5. Final Join back to original data
    df_result = df_hourly.merge(
        pivot_df,
        left_on=['sensorId', 'timestamp'],
        right_on=['sensor_id', 'timestamp'],
        how='left'
    ).drop(columns=['sensor_id'])

    return df_result

In [15]:
forecast_df.drop(columns='wind_direction_10m',inplace=True)
forecast_df.columns

Index(['timestamp', 'sensorId', 'lat', 'lon', 'temperature_2m',
       'relative_humidity_2m', 'wind_speed_10m', 'surface_pressure'],
      dtype='object')

In [16]:
forecast_df.rename(columns={"temperature_2m":"temperature","relative_humidity_2m":"humidity","surface_pressure":"pressure","wind_speed_10m":"wind_speed"},inplace=True)
forecast_df.columns

Index(['timestamp', 'sensorId', 'lat', 'lon', 'temperature', 'humidity',
       'wind_speed', 'pressure'],
      dtype='object')

In [17]:
weather_cols = ['humidity', 'pressure', 'temperature', 'wind_speed']

In [18]:
forecast_df = append_neighbors(forecast_df,neighbors_df_clean, weather_cols)
forecast_df

,timestamp,sensorId,lat,lon,temperature,humidity,wind_speed,pressure,neighbor1_humidity,neighbor2_humidity,...,neighbor1_temperature,neighbor2_temperature,neighbor3_temperature,neighbor4_temperature,neighbor5_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,neighbor4_wind_speed,neighbor5_wind_speed
0,2025-11-09 16:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.300,89.000000,2.200000,948.50000,86.0,86.0,...,14.0,14.1,14.1,14.0,13.5,3.0,3.0,3.0,3.0,1.8
1,2025-11-09 17:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.000,90.000000,3.600000,948.40000,92.0,92.0,...,13.3,13.4,13.4,13.4,13.1,5.2,5.2,5.2,5.2,1.3
2,2025-11-09 18:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.600,92.000000,2.200000,947.90000,95.0,95.0,...,12.9,13.0,13.0,13.0,12.7,4.0,4.0,4.0,4.0,2.2
3,2025-11-09 19:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.400,94.000000,1.800000,948.20000,95.0,95.0,...,12.8,12.9,12.9,12.8,12.6,4.4,4.4,4.4,4.4,1.8
4,2025-11-09 20:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.300,95.000000,2.800000,947.80000,94.0,94.0,...,12.7,12.8,12.8,12.8,12.5,4.7,4.7,4.7,4.7,0.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501079,2026-03-01 17:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,5.438,71.722170,1.548418,995.15894,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
501080,2026-03-01 18:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,7.638,66.715480,1.310420,995.11540,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
501081,2026-03-01 19:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.788,69.456400,2.305125,995.31110,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
501082,2026-03-01 20:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.338,68.128060,1.913426,995.45435,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
def extract_time_features(df, timestamp_col='timestamp'):

    month = df[timestamp_col].dt.month

    df['season'] = np.select(
        [
            month.isin([12, 1, 2]),
            month.isin([3, 4, 5]),
            month.isin([6, 7, 8]),
            month.isin([9, 10, 11])
        ],
        [
            'winter',
            'spring',
            'summer',
            'autumn'
        ]
    )

    df['hour_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.hour / 24)


    df['month_sin'] = np.sin(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)

    df['day_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)
    df['day_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)


    df['is_weekend'] = df[timestamp_col].dt.dayofweek.isin([5, 6]).astype(int)

    df['is_heating_season'] = df[timestamp_col].dt.month.isin([11, 12, 1, 2, 3]).astype(int)

    return df

In [20]:
forecast_df['timestamp'] = pd.to_datetime(forecast_df['timestamp'],utc=True)
forecast_df = extract_time_features(forecast_df)
forecast_df

,timestamp,sensorId,lat,lon,temperature,humidity,wind_speed,pressure,neighbor1_humidity,neighbor2_humidity,...,neighbor5_wind_speed,season,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2025-11-09 16:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.300,89.000000,2.200000,948.50000,86.0,86.0,...,1.8,autumn,-0.866025,-5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
1,2025-11-09 17:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.000,90.000000,3.600000,948.40000,92.0,92.0,...,1.3,autumn,-0.965926,-2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
2,2025-11-09 18:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.600,92.000000,2.200000,947.90000,95.0,95.0,...,2.2,autumn,-1.000000,-1.836970e-16,-0.866025,0.5,-0.781831,0.62349,1,1
3,2025-11-09 19:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.400,94.000000,1.800000,948.20000,95.0,95.0,...,1.8,autumn,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
4,2025-11-09 20:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.300,95.000000,2.800000,947.80000,94.0,94.0,...,0.7,autumn,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501079,2026-03-01 17:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,5.438,71.722170,1.548418,995.15894,NaN,NaN,...,NaN,spring,-0.965926,-2.588190e-01,0.866025,0.5,-0.781831,0.62349,1,1
501080,2026-03-01 18:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,7.638,66.715480,1.310420,995.11540,NaN,NaN,...,NaN,spring,-1.000000,-1.836970e-16,0.866025,0.5,-0.781831,0.62349,1,1
501081,2026-03-01 19:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.788,69.456400,2.305125,995.31110,NaN,NaN,...,NaN,spring,-0.965926,2.588190e-01,0.866025,0.5,-0.781831,0.62349,1,1
501082,2026-03-01 20:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,6.338,68.128060,1.913426,995.45435,NaN,NaN,...,NaN,spring,-0.866025,5.000000e-01,0.866025,0.5,-0.781831,0.62349,1,1


In [21]:
start = pd.Timestamp("2025-11-09 16:00:00",tz="UTC")
end = pd.Timestamp("2025-12-01 00:00:00",tz="UTC")

filtered_df = forecast_df[
    (forecast_df["timestamp"] >= start) &
    (forecast_df["timestamp"] < end)
]

In [22]:
filtered_df

,timestamp,sensorId,lat,lon,temperature,humidity,wind_speed,pressure,neighbor1_humidity,neighbor2_humidity,...,neighbor5_wind_speed,season,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2025-11-09 16:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.300,89.000000,2.200000,948.50000,86.0,86.0,...,1.8,autumn,-0.866025,-5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
1,2025-11-09 17:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,12.000,90.000000,3.600000,948.40000,92.0,92.0,...,1.3,autumn,-0.965926,-2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
2,2025-11-09 18:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.600,92.000000,2.200000,947.90000,95.0,95.0,...,2.2,autumn,-1.000000,-1.836970e-16,-0.866025,0.5,-0.781831,0.62349,1,1
3,2025-11-09 19:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.400,94.000000,1.800000,948.20000,95.0,95.0,...,1.8,autumn,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
4,2025-11-09 20:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,11.300,95.000000,2.800000,947.80000,94.0,94.0,...,0.7,autumn,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498897,2025-11-30 19:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,4.700,73.000000,2.800000,985.90000,NaN,NaN,...,NaN,autumn,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
498898,2025-11-30 20:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,4.300,78.000000,3.300000,986.20000,NaN,NaN,...,NaN,autumn,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
498899,2025-11-30 21:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,4.000,81.000000,2.600000,986.60000,NaN,NaN,...,NaN,autumn,-0.707107,7.071068e-01,-0.866025,0.5,-0.781831,0.62349,1,1
498900,2025-11-30 22:00:00+00:00,sensor_dev_8550_72,42.000000,21.380000,4.638,88.733864,0.763675,986.44727,NaN,NaN,...,NaN,autumn,-0.500000,8.660254e-01,-0.866025,0.5,-0.781831,0.62349,1,1


In [23]:
print("Loading Chronos-2 and generating forecasts...")
pipeline = Chronos2Pipeline.from_pretrained(
    "chronos2_zero_shot"
)

Loading Chronos-2 and generating forecasts...


In [24]:
filtered_df['humidity'] = filtered_df['humidity'].astype(float)

/tmp/ipykernel_356086/1051234344.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['humidity'] = filtered_df['humidity'].astype(float)


In [25]:
filtered_df['sensorId'].unique()

array(['007f2b03-94e6-47b3-9e3e-44273354acd5',
       '01440b05-255d-4764-be87-bdf135f32289',
       '01cf1cec-bf2d-41b3-8cd5-e8bd720f01b4',
       '07b58ccf-7faa-4f0a-a10a-e7b485d52ffe',
       '089aa02d-203e-4462-a395-b7dd86692480',
       '0a058579-12c9-47be-971b-607198002d3b',
       '0c33bee5-1139-472a-aef7-6d855bc5010d',
       '0e543e25-8bc3-4ce8-adbe-998e0d90e019',
       '0f10deea-03bc-4a47-ae87-85442140467c',
       '0f241923-b5b5-4c4a-bf08-279b9e3a8540', '1000', '1001', '1002',
       '1003', '1004', '1005', '10661e7e-e9c5-4e67-a5fa-06cdf6f3e76d',
       '10968604-7c7a-4c7b-82a1-937793ac214f',
       '11888f3a-bc5e-4a0c-9f27-702984decedf',
       '1286fb13-a4de-44cd-a390-4117fcddf1a9',
       '12bea5d2-b3a0-4c69-b970-80c3a9d0cc0b',
       '1587b09d-d0b0-4f46-b05e-d2fb93fc7863',
       '1a2af884-336b-427d-9b37-fe332557539f',
       '1f5c7035-6c1f-45ee-97f4-0a292b49710a',
       '1fc02db8-38d2-4bb5-a2cb-a72f357d0e51',
       '200cdb67-8dc5-4dcf-ac62-748db636e04e',
       '24ea

In [26]:
context_df['sensorId'].unique()

array(['007f2b03-94e6-47b3-9e3e-44273354acd5',
       '01cf1cec-bf2d-41b3-8cd5-e8bd720f01b4',
       '07b58ccf-7faa-4f0a-a10a-e7b485d52ffe',
       '0a058579-12c9-47be-971b-607198002d3b',
       '0f10deea-03bc-4a47-ae87-85442140467c', '1000', '1001', '1002',
       '1003', '1004', '10661e7e-e9c5-4e67-a5fa-06cdf6f3e76d',
       '11888f3a-bc5e-4a0c-9f27-702984decedf',
       '1286fb13-a4de-44cd-a390-4117fcddf1a9',
       '1a2af884-336b-427d-9b37-fe332557539f',
       '200cdb67-8dc5-4dcf-ac62-748db636e04e',
       '24eaebc2-ca62-49ff-8b22-880bc131b69f',
       '2c275060-799f-46c9-a2eb-fc613c4b4ff6',
       '35bdd494-5395-4ac5-b7b9-05b82c9b6acd',
       '3791738b-bec6-452e-9aae-5b11a899bfe2',
       '3d5e32fb-3c41-427f-b6ef-ed19e84026dd',
       '3d7bd712-24a9-482c-b387-a8168b12d3f4',
       '3e2465de-c2c2-4473-9c62-7113265debd9',
       '52546b6e-77bf-40f8-b0e5-6e54d7947e9f',
       '59c15198-7ba8-45af-aac4-5b3d1560fd81',
       '5de2a490-05f1-4f33-927e-a7e6f5664b72',
       '5e8e87e4-dd6

In [27]:
future_df = filtered_df[
    filtered_df["sensorId"].isin(context_df["sensorId"].unique())
].copy()

In [28]:
max(future_df['timestamp'])

Timestamp('2025-11-30 23:00:00+0000', tz='UTC')

In [29]:
future_df.drop(columns=['lat','lon'],inplace=True)

In [30]:
forecast_df = pipeline.predict_df(
    df=context_df,
    prediction_length=PREDICTION_LENGTH,
    target=TARGET,
    id_column=ID_COL,
    future_df = future_df,
    validate_inputs=False
)

In [31]:
# Merge predictions with actual values to align them
eval_df = pd.merge(
    forecast_df[[ID_COL, TIME_COL, 'predictions']],
    ground_truth_df[[ID_COL, TIME_COL, TARGET]],
    on=[ID_COL, TIME_COL]
)

In [32]:
eval_df.columns

Index(['sensorId', 'timestamp', 'predictions', 'pm25'], dtype='object')

In [33]:
eval_df

,sensorId,timestamp,predictions,pm25
0,007f2b03-94e6-47b3-9e3e-44273354acd5,2025-11-09 16:00:00+00:00,2.567884,3.50
1,007f2b03-94e6-47b3-9e3e-44273354acd5,2025-11-09 17:00:00+00:00,2.649847,2.00
2,007f2b03-94e6-47b3-9e3e-44273354acd5,2025-11-09 18:00:00+00:00,2.680747,2.00
3,007f2b03-94e6-47b3-9e3e-44273354acd5,2025-11-09 19:00:00+00:00,2.823944,1.75
4,007f2b03-94e6-47b3-9e3e-44273354acd5,2025-11-09 20:00:00+00:00,2.891205,2.25
...,...,...,...,...
38395,sensor_dev_84941_208,2025-11-30 19:00:00+00:00,4.807501,NaN
38396,sensor_dev_84941_208,2025-11-30 20:00:00+00:00,5.050131,NaN
38397,sensor_dev_84941_208,2025-11-30 21:00:00+00:00,4.973124,NaN
38398,sensor_dev_84941_208,2025-11-30 22:00:00+00:00,4.918943,NaN


In [34]:

from pathlib import Path
import sqlite3

DB_PATH = Path("../data/skopje.db")
if not DB_PATH.exists():
    DB_PATH = Path("data/skopje.db")

if not DB_PATH.exists():
    raise FileNotFoundError("Could not find data/skopje.db. Run the notebook from offline-Phase or the project root.")

CITY = "Skopje"
MODEL_VERSION = f"chronos2_{TARGET}_skopje_offline_test_{PREDICTION_LENGTH}h_zero_shot"
MODEL_TYPE = 'zero_shot'
train_df = (
    context_df.rename(
        columns={
            ID_COL: "sensor_id",
            TARGET: "actual_value"
        }
    )
)
train_df['predicted_value'] = np.nan
test_df = (
    eval_df
    .rename(columns={
        ID_COL:"sensor_id",
        TARGET: "actual_value",
        "predictions": "predicted_value",
    })
)
offline_results = pd.concat([train_df,test_df],ignore_index=True).sort_values(["sensor_id", "timestamp"])
offline_results = offline_results[["sensor_id", "timestamp", "actual_value", "predicted_value"]].copy()
offline_results["city"] = CITY
offline_results["pollutant"] = TARGET
offline_results["model_version"] = MODEL_VERSION
offline_results["model_type"] = MODEL_TYPE
offline_results["timestamp"] = pd.to_datetime(offline_results["timestamp"]).dt.strftime("%Y-%m-%d %H:%M:%S")
offline_results = offline_results[["city", "sensor_id", "timestamp", "pollutant", "actual_value", "predicted_value", "model_version","model_type"]]

records = list(offline_results.itertuples(index=False, name=None))

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS offline_test_results (
            city TEXT NOT NULL,
            sensor_id TEXT NOT NULL,
            timestamp TEXT NOT NULL,
            pollutant TEXT NOT NULL,
            actual_value REAL,
            predicted_value REAL,
            model_version TEXT NOT NULL,
            model_type TEXT NOT NULL,
            PRIMARY KEY (city, sensor_id, timestamp, pollutant, model_version)
        )
    """)
     # 2. Add Secondary Indexes for fast querying
     # Fast filtering by time range (e.g. WHERE timestamp >= '2025-11-09')
    conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_offline_timestamp
            ON offline_test_results (timestamp)
    """)

    # Fast filtering by sensor + timestamp queries
    conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_offline_sensor_time
            ON offline_test_results (sensor_id, timestamp)
    """)

    # Fast model performance evaluations (e.g. comparing model versions for a pollutant)
    conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_offline_model_eval
            ON offline_test_results (model_version, pollutant)
    """)
    conn.executemany("""
        INSERT OR REPLACE INTO offline_test_results (
            city, sensor_id, timestamp, pollutant, actual_value, predicted_value, model_version,model_type
        ) VALUES (?, ?, ?, ?, ?, ?, ?,?)
    """, records)

print(f"Saved {len(records)} {CITY} {TARGET} offline test rows to {DB_PATH}")

/tmp/ipykernel_356086/3198710792.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  offline_results = pd.concat([train_df,test_df],ignore_index=True).sort_values(["sensor_id", "timestamp"])


Saved 1315800 Skopje pm25 offline test rows to ../data/skopje.db


In [35]:
eval_df.dropna(axis=0,inplace=True)

In [36]:
eval_df.isnull().sum()

sensorId       0
timestamp      0
predictions    0
pm25           0
dtype: int64

In [37]:
y_true = eval_df['pm25']
y_pred = eval_df['predictions']

In [38]:
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"--- Global Model Evaluation ---")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}")

--- Global Model Evaluation ---
RMSE: 13.2534
R2:   0.1747


In [39]:
per_sensor = eval_df.groupby(ID_COL).apply(
    lambda df: pd.Series({
        "r2": r2_score(df["pm25"], df["predictions"]),
        "rmse": np.sqrt(mean_squared_error(df["pm25"], df["predictions"]))
    }),
    include_groups=False
)

print(per_sensor)

                                             r2       rmse
sensorId                                                  
007f2b03-94e6-47b3-9e3e-44273354acd5  -0.030675   2.169156
01cf1cec-bf2d-41b3-8cd5-e8bd720f01b4  -0.897117  11.551052
0a058579-12c9-47be-971b-607198002d3b  -0.027488   9.356143
0f10deea-03bc-4a47-ae87-85442140467c   0.014807   5.077621
1000                                  -0.078726  16.422743
1001                                   0.180420  18.715368
1002                                   0.060016  16.857751
1003                                   0.035608  18.903695
1004                                 -12.959416  18.268266
10661e7e-e9c5-4e67-a5fa-06cdf6f3e76d   0.059939   8.199437
11888f3a-bc5e-4a0c-9f27-702984decedf   0.013494  14.428051
1286fb13-a4de-44cd-a390-4117fcddf1a9  -0.118534   7.243558
1a2af884-336b-427d-9b37-fe332557539f   0.105994  20.317940
200cdb67-8dc5-4dcf-ac62-748db636e04e   0.154048  19.531965
24eaebc2-ca62-49ff-8b22-880bc131b69f  -0.038367   8.9780